### Indexing


In [6]:
%pip install --upgrade \
numpy \
pandas \
scipy \
scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# downloading the document
import wget

file_name = "company_policy.txt"
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt"

wget.download(url, out=file_name)
print("file downloaded")

file downloaded


In [8]:
with open(file_name, "r") as f:
    content = f.read()
    print(content)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [9]:
# Document Loading & Chunking
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(file_name)
document = loader.load()
# print(document[0].page_content)

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(document)
print(chunks)

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


[Document(metadata={'source': 'company_policy.txt'}, page_content='1.\tCode of Conduct'), Document(metadata={'source': 'company_policy.txt'}, page_content="Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decision

In [10]:
# Embedding and Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding = HuggingFaceEmbeddings()
vector_store = Chroma.from_documents(chunks, embedding)
print("document  ingested")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 246.66it/s]


document  ingested


### LLM Model Construction


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OLLAMA_API_KEY")

if not API_KEY:
    raise ValueError("API KEY NOT FOUND")

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="gemma4:31b-cloud",
    base_url="https://ollama.com",
    search_kwargs={"headers": {"Authorization": f"Bearer {API_KEY}"}},
    temperature=0,
    num_predict=256,
)

print("llm finetuned sucessfully!")
# Ollama does not expose a separate decoding_method="greedy" parameter in OllamaLLM.

llm finetuned sucessfully!


### Retrieval


In [12]:
retriever = vector_store.as_retriever()

from langchain_classic.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=False
)

query1 = "What is mobile policy?"
query2 = "Can I eat in company vehicles?"
# resp = qa.invoke(query1)
resp = qa.invoke(query2)
print(resp["result"])

I don't know.


### Using Prompt Template


In [13]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a company policy assistant.

Answer questions using only the provided context.

If the context does not contain the answer,
clearly state that the company policy does not specify it.

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}  # Specified Prompt as a chain type keyward args

In [14]:
# Update the qa

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False,
    chain_type_kwargs=chain_type_kwargs,
)

query = "Can I eat in company vehicles?"
resp = qa.invoke(query)
print(resp)

{'query': 'Can I eat in company vehicles?', 'result': 'The company policy does not specify it.'}


### Make Memory and Chain


In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(
    memory_key="chat_history", return_messages=True, output_key="answer"
)

chatbot =   .from_llm(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    memory=memory,
    return_source_documents=False,
    combine_docs_chain_kwargs=chain_type_kwargs,
)


def conversation():
    while True:
        query = input("Question: ").strip()

        if query.lower() in ["quit", "exit", "bye", "goodbye"]:
            print("Answer: Goodbye!")
            break

        if query.lower() == "history":
            get_chat_history()
            continue

        if not query:
            continue

        result = chatbot.invoke({"question": query})

        print("Answer:", result["answer"])


def get_chat_history():
    print(len(memory.chat_memory.messages))
    for message in memory.chat_memory.messages:
        print(message.type, ":", message.content)

        print()

In [38]:
conversation()

Answer: The Code of Conduct outlines the fundamental principles and ethical standards of the organization, focusing on the following:

*   **Integrity:** Acting honestly and transparently, protecting sensitive information, and avoiding conflicts of interest.
*   **Respect:** Embracing diversity, valuing individual contributions, and prohibiting discrimination, harassment, or disrespectful behavior to ensure an inclusive environment.
*   **Accountability:** Taking responsibility for actions and decisions, following laws and regulations, reporting potential violations, and supporting investigations.
*   **Safety:** Prioritizing the safety of employees, clients, and communities, and reporting unsafe conditions or practices.
*   **Environmental Responsibility:** Committing to sustainable practices and minimizing the organization's environmental footprint.
Answer: The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices to

In [39]:
get_chat_history()

6
human : code of conduct

ai : The Code of Conduct outlines the fundamental principles and ethical standards of the organization, focusing on the following:

*   **Integrity:** Acting honestly and transparently, protecting sensitive information, and avoiding conflicts of interest.
*   **Respect:** Embracing diversity, valuing individual contributions, and prohibiting discrimination, harassment, or disrespectful behavior to ensure an inclusive environment.
*   **Accountability:** Taking responsibility for actions and decisions, following laws and regulations, reporting potential violations, and supporting investigations.
*   **Safety:** Prioritizing the safety of employees, clients, and communities, and reporting unsafe conditions or practices.
*   **Environmental Responsibility:** Committing to sustainable practices and minimizing the organization's environmental footprint.

human : mobile policy

ai : The Mobile Phone Policy sets forth the standards and expectations for the appropria